In [ ]:
import numpy as np 
from pathlib import Path 
import librosa 
import soundfile as sf
import pickle
import pandas as pd 
from IPython.display import Audio

In [ ]:
## Make manifest to run cued version of saddler word rec experiment 

# paths to all SWC data 
fn_pkl_src = '/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl'
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'

# path to foregrounds used in online experiment 
path_to_stim = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/")
foreground_dir = path_to_stim / "foreground_swc"

fg_manifest = pd.read_pickle(foreground_dir / "manifest.pdpkl")

In [ ]:
# get cues by matching rows to those in binaural experiment df 

out_path = Path('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/')
binaural_df = pd.read_pickle(out_path / 'full_eval_trial_manifest_new_fnames.pdpkl')

In [ ]:
binaural_df.head()

In [ ]:
## Select cue excerpts from words not included in online experiment

np.random.seed(0)
words = fg_manifest.word.unique()
all_words_not_filtered = pd.read_pickle(fn_pkl_dst)
oov_excerpts = all_words_not_filtered[~all_words_not_filtered['word'].isin(words)]

# get cue excerpts from oov excerpts 
talkers = fg_manifest['client_id'].unique()
samples_per_talker = {talker:count for talker,count in fg_manifest.client_id.value_counts().items()}
viables_cues = oov_excerpts[oov_excerpts.client_id.isin(talkers)]

cues = viables_cues.groupby('client_id').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='client_id', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'cue_src_ix'}, inplace=True)

# group and combine 

fg_manifest.sort_values(by='client_id', inplace=True)
fg_manifest.reset_index(inplace=True, drop=True)

cues.sort_values(by='client_id', inplace=True)
cues.reset_index(inplace=True, drop=True)


### Merge cues with foregrounds  
cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']] = cues[['word', 'cue_src_ix', 'client_id', 'src_fn', 'clip_start_in_s', 'clip_end_in_s']]
# Combine as experiment dataframe
tg_and_cue_df = fg_manifest.join(cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']])
assert (tg_and_cue_df['client_id'] == tg_and_cue_df['cue_client_id']).all(), "Cue and Target talkers don't match!"



In [ ]:
tg_and_cue_df

In [ ]:
def pad_or_trim_to_len(x, n, mode='both', kwargs_pad={}):
    """
    Increases or decreases the length of a one-dimensional signal
    by either padding or triming the array. If the difference
    between `len(x)` and `n` is odd, this function will default to
    adding/removing the extra sample at the end of the signal.
    
    Args
    ----
    x (np.ndarray): one-dimensional input signal
    n (int): length of output signal
    mode (str): specify which end of signal to modify
        (default behavior is to symmetrically modify both ends)
    kwargs_pad (dict): keyword arguments for np.pad function
    
    Returns
    -------
    x_out (np.ndarray): one-dimensional signal with length `n`
    """
    assert len(np.array(x).shape) == 1, "input must be 1D array"
    assert mode.lower() in ['both', 'start', 'end']
    n_diff = np.abs(len(x) - n)
    if len(x) > n:
        if mode.lower() == 'end':
            x_out = x[:n]
        elif mode.lower() == 'start':
            x_out = x[-n:]
        else:
            x_out = x[int(np.floor(n_diff / 2)):-int(np.ceil(n_diff / 2))]
    elif len(x) < n:
        if mode.lower() == 'end':
            pad_width = [0, n_diff]
        elif mode.lower() == 'start':
            pad_width = [n_diff, 0]
        else:
            pad_width = [int(np.floor(n_diff / 2)), int(np.ceil(n_diff / 2))]
        kwargs = {'mode': 'constant'}
        kwargs.update(kwargs_pad)
        x_out = np.pad(x, pad_width, **kwargs)
    else:
        x_out = x
    assert len(x_out) == n
    return x_out
 

def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    load_full_file = True
    total_file_duration_in_s = librosa.get_duration(filename=dfi.src_fn)
    if (clip_start_in_s >= 0) and (clip_end_in_s < total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

In [ ]:
## write cue excerpts and update path 
from tqdm.auto import tqdm
import soxr


sr = 44100
duration = 3

# write cue excerpts to new directory
out_dir = Path("/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00/cue_excerpts")
# make dir and parennts 
out_dir.mkdir(parents=True, exist_ok=True)

# create cue excerpt and update manifest 
for row in tqdm(tg_and_cue_df.itertuples(), total=len(tg_and_cue_df)):
    cue_excerpt = get_excerpt(row, dur=duration, sr=sr, pad_with_context=True, jitter_fraction=0)
    cue_excerpt_fn = out_dir / f"{row.cue_client_id}.wav"
    sf.write(cue_excerpt_fn, cue_excerpt, samplerate=sr)
    tg_and_cue_df.loc[row.Index, 'cue_src_fn'] = str(cue_excerpt_fn)
    # break



In [ ]:
# save for expmnt
out_dir = Path("/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00")

tg_and_cue_df.to_pickle(out_dir / 'cue_and_target_manifest.pdpkl')

In [ ]:
## load WSN vocab mapping 
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']
word_2_class = {v:k for k,v in class_map.items()}

In [ ]:
Audio(fg_manifest['src_fn'][0])

In [ ]:
# fg_manifest['word'].map(word_2_class)

test_condition_dict = {'music':"background_musdb18hq",
                       "babble":"background_cv08talkerbabble",
                       "stationary": "background_issnstationary",
                       "modulated": "background_issnfestenplomp",
                       "audioset": "background_audioset",
                       "ieee_scenes": "background_ieeeaaspcasa"}

In [ ]:
Audio(path_to_stim / "background_audioset"/"021.wav" )

In [ ]:
# write torch dataloader for test set 


import torch
import pandas as pd 
from pathlib import Path

class SaddlerSWCWordRecTest(torch.utils.data.Dataset):
    """
    Dataset for the Saddler word recognition experiment using
    foreground excerpts from Spoken Wikipedia.  Backgrounds are 
    either: 
        - music from musdb
        - 8 talkerbabble from common voice
        - spectrally matched noise (SSN; match each foreground)
        - Festen and Plomp style modulated maskers
        - audioset
        - natural scenes from IEEE AASP CASA challenge
        - clean (no background)
    """
    def __init__(self, manifest_path, bg_stim_path, condition, label_type="WSN", sr=20_000):
        """
        Args:
            manifest_path (str): path to pandas manifest with fg and cue excerpts
            bg_stim_path (str): path to directory with background stimuli
            condition (str): background condition. Either "music", "babble", "stationary", "modulated", "audioset", "ieee_scenes", or "clean"
            label_type (str): Set of word class labels to use. Either "WSN" (JSIN) or "CV" common voice
            sr (int): sampling rate to load audio at
        """ 
        self.manifest = pd.read_pickle(manifest_path)
        self.condition = condition
        self.label_type = label_type
        self.sr = sr 
        self.condition_dict = {'music':"background_musdb18hq",
                       "babble":"background_cv08talkerbabble",
                       "stationary": "background_issnstationary",
                       "modulated": "background_issnfestenplomp",
                       "audioset": "background_audioset",
                       "ieee_scenes": "background_ieeeaaspcasa",
                       }
        
        if condition == "clean":
            self.bg_stim = None
            self.test_cond_dir = None

        else:
        
            self.test_cond_dir = self.condition_dict[self.condition]
            self.bg_stim = list((bg_stim_path / self.test_cond_dir).glob("*.wav"))
        self.class_map, self.word_2_class = self.class_map()

        self.dataset_len = len(self.manifest)

    def class_map(self):
        """
        Loads the mapping between the word IDX and human readable word map. 
        """
        if self.label_type == "WSN":
            ## load WSN vocab mapping 
            word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
            class_map = word_and_speaker_encodings['word_idx_to_word']
        elif self.label_type == "CV":
            class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_word_int_label_dict.pkl", "rb" )) 
        word_2_class = {v:k for k,v in class_map.items()}
        return class_map, word_2_class

    def __getitem__(self, index):
        """
        Gets components of the hdf5 file that are used for training
        Args: 
            index (int): index into the hdf5 file
        Returns:
            [signal, target] : the training audio (signal) containing the preprocessing
            which may combine the foreground and background speech, and the target idx
            specified by target_keys. 
        """
        foreground, _ = librosa.load(self.manifest['src_fn'][index], sr=self.sr)
        cue, _ = librosa.load(self.manifest['cue_src_fn'][index], sr=self.sr)
        if self.condition == "clean":
            background = None
        else:
            background, _  = librosa.load(self.bg_stim[index], sr=self.sr)
    
        word = self.manifest['word'][index]
        word_label = self.word_2_class[word]
    
        return cue, foreground, background, word_label

    def __len__(self):
        return self.dataset_len


In [ ]:
# try dataset 
out_dir = Path("/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00")

path_to_stim = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/")

dataset = SaddlerSWCWordRecTest(manifest_path=out_dir / 'cue_and_target_manifest.pdpkl',
                            bg_stim_path=path_to_stim,
                            condition='clean',
                            label_type="WSN", sr=20_000)

In [ ]:
## Make test condition manifest
import itertools

# snrs are -12 to + 6 dB in 3 dB steps

snrs = np.arange(-12, 4, 3)

# get conditiosn 
conditions = list(test_condition_dict.keys())

# all pairings of snrs and conditions 
condition_pairs = list(itertools.product(conditions, snrs))

condition_dict = dict(zip(range(len(condition_pairs)), condition_pairs))
# add new condition 
condition_dict[len(condition_dict)] = ('clean', 'clean')
condition_dict

In [ ]:
## Save out as pickle 

import pickle

out_name = "/om2/user/imgriff/projects/Auditory-Attention/saddler_test_condition_dict.pkl"

#write pickle 
with open(out_name, 'wb') as handle:
    pickle.dump(condition_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)   

In [26]:
!cd Auditory-Attention  
import torch 
import sys 
sys.path.append('Auditory-Attention')
import numpy as np 
from tqdm.auto import tqdm
from pytorch_lightning import seed_everything
from src.attn_tracking_lightning import AttentionalTrackingModule
from src.spatial_attn_lightning import BinauralAttentionModule 
from corpus.saddler_word_rec import SaddlerSWCWordRecTest
import src.audio_transforms as at
import scipy.stats as stats
import yaml
from pathlib import Path

condition = 'clean'
snr = 'clean'

config = Path("Auditory-Attention/config/attentional_cue/attn_cue_lr_1e-4_bs_64_constrained_slope_multi_distractor.yaml")



model_name = config.stem
checkpoint_path = "Auditory-Attention/attn_cue_models/attn_cue_jsin_multi_distractor_w_audioset_bs_64_lr_1e-4/checkpoints/epoch=0-step=70000.ckpt"
config = yaml.load(open(config, 'r'), Loader=yaml.FullLoader)
print(f"Evaluating {model_name} on {condition} at {snr}db SNR")
print(f"Loading model from {checkpoint_path}")


module = AttentionalTrackingModule
config['data']['audio']['rep_kwargs']['center_crop'] = True
config['data']['audio']['rep_kwargs']['out_dur'] = 2


label_type = "WSN"
sr = 20_000

# load and freeze model
model = module.load_from_checkpoint(checkpoint_path=checkpoint_path, config=config).eval()


def collate_fn(batch):
    #apply transforsms to batch
    cues = torch.stack([audio_transforms(cue, None)[0] for cue, _, _, _ in batch])
    mixtures = torch.stack([audio_transforms(fg,bg)[0] for _, fg, bg, _ in batch])
    labels = torch.tensor([label for _, _, _, label in batch]).type(torch.LongTensor)
    return cues, mixtures, labels



dataset = SaddlerSWCWordRecTest(manifest_path="/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00/cue_and_target_manifest.pdpkl",
                bg_stim_path="/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/",
                condition=condition,
                label_type=label_type,
                sr=sr)

dataloader = torch.utils.data.DataLoader(dataset,
                        batch_size=1,
                        shuffle=False,                
                        collate_fn=collate_fn,
                        num_workers=0)

audio_config = config['data']['audio']
IIR_COCH = not audio_config['rep_kwargs']['rep_on_gpu']


if IIR_COCH:
    audio_transforms = at.AudioCompose([
        at.AudioToTensor(),
        at.CombineWithRandomDBSNR(low_snr=snr, high_snr=snr), 
        at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
        at.UnsqueezeAudio(dim=0),
        at.AudioToAudioRepresentation(**audio_config),
        # at.UnsqueezeAudio(dim=0),
    ])
else:
    audio_transforms = at.AudioCompose([
        at.AudioToTensor(),
        at.CombineWithRandomDBSNR(low_snr=snr, high_snr=snr), 
        at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
        at.UnsqueezeAudio(dim=0),
    ])  


Evaluating attn_cue_lr_1e-4_bs_64_constrained_slope_multi_distractor on clean at cleandb SNR
Loading model from Auditory-Attention/attn_cue_models/attn_cue_jsin_multi_distractor_w_audioset_bs_64_lr_1e-4/checkpoints/epoch=0-step=70000.ckpt
ln_first
center_crop=True
binaural=False
using IIR cochleagram
center_crop=True
binaural=False
using IIR cochleagram


In [27]:
model.cuda()
# run eval loop
results = []
# i = 100
for ix, batch in enumerate( tqdm(dataloader)):
    cue, mixture, word = batch

    # to device 
    cue = cue.cuda()
    mixture = mixture.cuda()
    logits = model(cue, mixture)
    preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach()
    results.extend(word == preds)
    if ix == 2:
        break

res_err = stats.sem(results)
res = np.mean(results)

  0%|          | 0/376 [00:00<?, ?it/s]

tensor([169])